# Bernacchi 2013 C3 photosynthesis model

Reference: Carl J. Bernacchi, Justin E. Bagley, Shawn P. Serbin, Ursula M. Ruiz-Vera, David M. Rosenthal, Andy VanLoocke  
"Modelling C3 photosynthesis from the chloroplast to the ecosystem"  
Plant, Cell & Environment 36.9 (2013): 1641-1657.  

[doi: 10.1111/pce.12118](https://www.doi.org/10.1111/pce.12118)

Unscanned conditions are those of the paper's Fig 2 (Ci 270 µmol mol⁻¹, PPFD 1500 µmol m⁻² s⁻¹, 25 °C), except that Figs 3 and 4 hold Ci at `CI_FIG34` (247.4 µmol mol⁻¹): their Rubisco-limited rate at 25 °C (19.8) lies below Fig 2's at 270 (21.6), so the captions' Ci of 270 cannot reproduce them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mxlmodels import get_bernacchi_2013
from mxlmodels.ss.bernacchi2013 import CI_FIG34

COLUMNS = ["rubisco", "rubp", "tpu", "an"]


def response(**conditions: np.ndarray | float) -> pd.DataFrame:
    """Limiting rates and net assimilation, indexed by the one scanned condition."""
    defaults = {"Ci": 270.0, "PPFD": 1500.0, "T_leaf": 25.0}
    (name, x), = ((k, v) for k, v in conditions.items() if np.ndim(v) == 1)
    rates = get_bernacchi_2013(**(defaults | conditions))
    return pd.DataFrame(dict(zip(COLUMNS, rates)), index=pd.Index(x, name=name))


def plot_response(res: pd.DataFrame, title: str, xlabel: str, xlim, ylim, xticks, yticks):
    fig, ax = plt.subplots(figsize=(6.2, 4.6))
    ax.plot(res.index, res["rubisco"], "--", color="royalblue", label="Rubisco")
    ax.plot(res.index, res["rubp"], "--", color="mediumvioletred", label="RuBP")
    ax.plot(res.index, res["tpu"], "--", color="darkkhaki", label="TPU")
    ax.plot(res.index, res["an"], color="black", linewidth=2.5, label=r"$A$")
    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=r"$A$ ($\mu$mol m$^{-2}$ s$^{-1}$)",
        xlim=xlim,
        ylim=ylim,
        xticks=xticks,
        yticks=yticks,
    )
    ax.legend(frameon=False, ncol=4, fontsize=8)
    fig.tight_layout()
    return fig, ax

## Figure 2: photosynthetic CO₂ response

In [ ]:
fig2 = response(Ci=np.linspace(1, 1000, 700))
fig, ax = plot_response(
    fig2,
    r"Photosynthetic-CO$_2$ response curve",
    r"$C_i$ ($\mu$mol mol$^{-1}$)",
    (0, 1000),
    (-10, 55),
    np.arange(0, 1001, 200),
    np.arange(-10, 51, 10),
)
ax.axhline(0, color="0.75", linewidth=1)
fig.savefig("figs/fig2.svg")
plt.show()

## Figure 3: photosynthetic leaf-temperature response

In [ ]:
fig3 = response(T_leaf=np.linspace(5, 40, 351), Ci=CI_FIG34)
fig, ax = plot_response(
    fig3,
    r"Photosynthetic-$T_{\mathrm{leaf}}$ response curve",
    r"$T_{\mathrm{leaf}}$ ($^\circ$C)",
    (5, 40),
    (0, 35),
    np.arange(5, 41, 5),
    np.arange(0, 36, 5),
)
fig.savefig("figs/fig3.svg")
plt.show()

## Figure 4: photosynthetic PPFD response

In [ ]:
fig4 = response(PPFD=np.linspace(0, 2000, 700), Ci=CI_FIG34)
fig, ax = plot_response(
    fig4,
    "Photosynthetic-PPFD response curve",
    r"PPFD ($\mu$mol m$^{-2}$ s$^{-1}$)",
    (0, 2000),
    (0, 35),
    np.arange(0, 2001, 500),
    np.arange(0, 36, 5),
)
fig.savefig("figs/fig4.svg")
plt.show()